In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common

cfg = common.load_config("dev")
cat = cfg["catalog"]
want = {"branches": 5, "loan_products": 8, "credit_policy": 12, "customers": 167,
        "loan_applications": 237, "credit_checks": 237, "loans": 127, "disbursements": 127,
        "bank_settlements": 127, "emi_schedule": 1953, "repayments": 679,
        "cash_deposit_slips": 34, "collections": 30}
current = {"loan_products": 4, "customers": 119}       # current rows of the history tables
got = {t: spark.table(f"{cat}.silver.{t}").count() for t in want}
got_current = {t: spark.table(f"{cat}.silver.{t}").filter("is_current").count() for t in current}
waiting = {(r["table_name"], r["rule"]): r["count"] for r in spark.table(f"{cat}.silver.quarantine")
           .filter("status = 'open'").groupBy("table_name", "rule").count().collect()}
unknown = spark.table(f"{cat}.silver.repayments").filter("orphan_of IS NOT NULL").count()
bank = {r["status"]: r["count"] for r in spark.table(f"{cat}.silver.bank_settlements").groupBy("status").count().collect()}
slips = spark.table(f"{cat}.silver.cash_deposit_slips")
s = slips.selectExpr("count(DISTINCT slip_no) AS slips", "sum(amount) AS total", "min(ocr_confidence) AS lowest").first()
display(slips.orderBy("slip_no", "branch_id").limit(8))
print(got)
print("current", got_current, "| waiting", waiting, "| repayments with unknown loan", unknown, "| bank", bank)
print("slips", s["slips"], "| total", s["total"], "| lowest OCR confidence", s["lowest"])
print("PASS" if got == want and got_current == current and unknown == 11 and bank == {"settled": 123, "returned": 4}
      and waiting == {("repayments", "orphan"): 3, ("credit_policy", "date_order"): 1}
      and s["slips"] == 11 and float(s["total"]) == 640453.99 else "FAIL")